# EVASPA Edge benchmark

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

In [ ]:
from sweat.common import filter, io
from sweat.evaspa.edge import Edge

Plot functions

In [ ]:
def plot(
    var: npt.NDArray,
    lst: npt.NDArray,
    edges: dict[str, Edge] | None = None,
    points: list[tuple[npt.NDArray, npt.NDArray]] | None = None,
    params: list[str] | None = None,
    var_name: str | None = None,
    out: str | None = None,
    density: bool | None = False,
):
    """
    Plot
    """
    # Figure
    _ = plt.figure(constrained_layout=True, figsize=(6, 5))

    ax = plt.gca()

    # plot all points
    if not density:
        ax.scatter(var, lst, s=1, alpha=1, color="moccasin", clip_on=False)
    else:
        # Flatten and mask values
        x = var.values.flatten()
        y = lst.values.flatten()
        mask = np.isfinite(x) & np.isfinite(y)
        x = x[mask]
        y = y[mask]
        ax.hist2d(x, y, (150, 150), cmap=plt.cm.viridis, cmin=1)

    # Fix limits
    xmin = np.nanmin(var) - 0.05
    xmax = np.nanmax(var) + 0.05
    ymin = np.nanmin(lst) - 5
    ymax = np.nanmax(lst) + 5
    ax.set_xlim([xmin, xmax])
    ax.set_ylim([ymin, ymax])

    # plot edge
    if points is not None:
        for pts in points:
            x_pts = pts[0]
            y_pts = pts[1]
            ax.scatter(x_pts, y_pts, fc="black", ec="w", marker="X", s=50)

    if edges is not None:
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)
        var_arr = np.linspace(var_min, var_max, num=1000)
        for pos, edge in edges.items():
            edge_arr = edge.get(var_arr)
            color = "red"
            if pos == "bottom":
                color = "blue"
            ax.plot(
                var_arr,
                edge_arr,
                color=color,
                linewidth=3,
                zorder=1,
            )

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    # Save
    if edges is not None:
        plot_title = "EF"
        cfg = (
            f"percentile={params[3]} (limit = {params[4]})"
            if params[2] == "percentile" and params[4] is not None
            else f"percentile = {params[3]}"
            if params[2] == "percentile"
            else (f"percentile={params[3]}/{params[4]}")
            if params[2] == "percentile_bounds"
            else f"nb_points={params[3]}"
        )

        if params is not None and len(params) == 5:
            plot_title = (
                f"LST-{var_name} \n EF {params[0]} interval={params[1]} {cfg}"
            )
        ax.set_title(plot_title, fontsize=10)
        if out is not None and params is not None and len(params) == 5:
            filename = f"ef_{params[0]}_interval_{params[1]}_{params[2]}_{params[3]}_{params[4]}.png"
            plt.savefig(os.path.join(out, filename))

    plt.show()

### Read data

In [ ]:
# Input data
src_path = os.path.dirname(os.path.dirname(os.getcwd()))
data = io.read_data_from_file(
    os.path.join(src_path, "tests", "data", "modis_test_geo.tif")
)
# Output dir
output_data = "out"

### Filter data

In [ ]:
filter_config = {
    "cloud": {"op": "==", "value": 0},  # keep clear pixels
    "water": {"op": "==", "value": 0},  # keep non water pixels
    "qa": {"op": "==", "value": 1},  # keep pixels with qa = 1
    "ndvi": {
        "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
    },  # Keep NDVI between [0,1]
    "fcover": {
        "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
    },  # Keep Fcover between [0,1]
    "albedo": {
        "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
    },  # Keep Fcover between [0,1]
    "height": {
        "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 300}]
    },  # Keep height between [0,300]
}

In [ ]:
valid_mask, _ = filter.find_valid_pixels(data=data, valid_config=filter_config)

In [ ]:
valid_mask.plot()

Keep valid pixels

In [ ]:
lst = data["lst"].where(valid_mask)
fcover = data["fcover"].where(valid_mask)
ndvi = data["ndvi"].where(valid_mask)
albedo = data["albedo"].where(valid_mask)

### Define plot edges

In [ ]:
def run(
    var_name,
    var,
    lst,
    position,
    models,
    intervals,
    interval_limits=["percentile(0.01)", "percentile(99.99)"],  # noqa
    points=None,
    percentiles=None,
    percentile_limits=None,
):
    """
    Method to run several edges, with interval_nb and nb_points
    """
    if position in {"top", "bottom"}:
        positions = [position]
    elif position == "both":
        positions = ["top", "bottom"]
    else:
        msg = "Position must be either 'top', 'bottom' or 'both'"
        raise ValueError(msg)
    if points is not None and percentiles is not None:
        msg = "Please provide points or percentiles"
        raise ValueError(msg)
    if points is not None:
        pts_selection = "nb_points"
        values_selection = points
    if percentiles is not None:
        pts_selection = "percentile"
        values_selection = percentiles
    out = os.path.join(output_data, var_name, position)
    os.makedirs(out, exist_ok=True)
    for name_model in models:
        use_breakpoint = False
        name = name_model
        if m := re.match(r"(.*)_bk", name):
            name = m.group(1)
            use_breakpoint = True
        for interval_nb in intervals:
            for value in values_selection:
                for percentile_limit in percentile_limits:
                    edges = {}
                    edge_pts = []
                    for pos in positions:
                        config = {
                            "position": pos,
                            "interval_type": "size",
                            "interval_nb": interval_nb,
                            "interval_limits": interval_limits,
                            f"{pts_selection}": value,
                            "percentile_limit": percentile_limit,
                            "selection": "median",
                            "use_breakpoint": use_breakpoint,
                        }
                        edge = Edge.create(name, config)
                        edge.fit(var, lst)
                        edges[pos] = edge
                        pts = edge.get_points(var, lst)
                        edge_pts.append(pts)
                    plot(
                        var,
                        lst,
                        edges=edges,
                        points=edge_pts,
                        params=[
                            f"{name}{'_bk' if use_breakpoint else ''}",
                            interval_nb,
                            pts_selection,
                            value,
                            percentile_limit,
                        ],
                        var_name=var_name,
                        out=out,
                    )

In [ ]:
def run_with_variable_percentiles(
    var_name,
    var,
    lst,
    position,
    models,
    percentile_bounds,
    percentile_intervals,
    interval_nb,
    interval_limits=["percentile(0.01)", "percentile(99.99)"],  # noqa
):
    """
    Method to run several edges, with various configuration
    """
    if position in {"top", "bottom"}:
        positions = [position]
    elif position == "both":
        positions = ["top", "bottom"]
    else:
        msg = "Position must be either 'top', 'bottom' or 'both'"
        raise ValueError(msg)
    out = os.path.join(output_data, var_name, position)
    os.makedirs(out, exist_ok=True)
    for name_model in models:
        use_breakpoint = False
        name = name_model
        if m := re.match(r"(.*)_bk", name):
            name = m.group(1)
            use_breakpoint = True
        edges = {}
        edge_pts = []
        for pos in positions:
            config = {
                "position": pos,
                "interval_type": "size",
                "interval_nb": interval_nb,
                "interval_limits": interval_limits,
                "percentile_bounds": percentile_bounds,
                "percentile_intervals": percentile_intervals,
                "selection": "median",
                "use_breakpoint": use_breakpoint,
            }
            edge = Edge.create(name, config)
            edge.fit(var, lst)
            edges[pos] = edge
            pts = edge.get_points(var, lst)
            edge_pts.append(pts)
        plot(
            var,
            lst,
            edges=edges,
            points=edge_pts,
            params=[
                f"{name}{'_bk' if use_breakpoint else ''}",
                interval_nb,
                "percentile_bounds",
                str(percentile_bounds),
                str(percentile_intervals),
            ],
            var_name=var_name,
            out=out,
        )

### Test LST-Fcover

Plot

In [ ]:
plot(fcover, lst, var_name="fcover", density=True)

Edges: 

- edges types: flat, linear regression &  parabolic
- number of intervals: 10
- interval limits defined with percentile ['percentile(0.01)','percentile(99.99)']
- percentiles: 2,5
- percentile limit (maximum number of points to take into account): None, 100 


In [ ]:
run(
    "fcover",
    fcover,
    lst,
    position="both",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    intervals=[10],
    interval_limits=[
        "percentile(0.01)",
        "percentile(99.99)",
    ],  # Interval filtering
    percentiles=[2, 5],
    percentile_limits=[None, 100],
)

In [ ]:
run_with_variable_percentiles(
    "fcover",
    fcover,
    lst,
    position="both",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    interval_nb=10,
    interval_limits=[
        "percentile(0.01)",
        "percentile(99.99)",
    ],  # Interval filtering
    percentile_bounds=[10, 0.01],
    percentile_intervals=[100, 1000000],
)

### Test LST-NDVI

Plot

In [ ]:
plot(ndvi, lst, var_name="ndvi", density=True)

Edges: 

- edges types: flat, linear regression &  parabolic
- number of intervals: 10
- interval limits defined with percentile ['percentile(0.01)','percentile(99.99)']
- percentiles: 2,5
- percentile limit (maximum number of points to take into account): None, 100 

In [ ]:
run(
    "ndvi",
    ndvi,
    lst,
    position="both",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    intervals=[10],
    interval_limits=[
        "percentile(0.01)",
        "percentile(99.99)",
    ],  # Interval filtering
    percentiles=[2, 5],
    percentile_limits=[None, 100],
)

In [ ]:
run_with_variable_percentiles(
    "ndvi",
    ndvi,
    lst,
    position="both",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    interval_nb=10,
    interval_limits=[
        "percentile(0.01)",
        "percentile(99.99)",
    ],  # Interval filtering
    percentile_bounds=[10, 0.01],
    percentile_intervals=[100, 1000000],
)

### Test LST-albedo

Plot

In [ ]:
plot(albedo, lst, var_name="albedo", density=True)

Edge:  

- edges types: 7 edges types to test for both wet and dry edge: flat, linear regression, linear regression (considering only albedo after breakpoint), parabolic, linear regression on albedo upper than the break point, linear regression on albedo upper than the break point and flat on lower albedos, double linear regression on albedo upper and lower than the break point
- number of intervals: 10
- interval limits defined with percentile ['percentile(0.01)','percentile(99.99)']
- percentiles: 2,5
- percentile limit (maximum number of points to take into account): None, 100 

In [ ]:
run(
    "albedo",
    albedo,
    lst,
    position="both",
    models=[
        "FlatRegressionEdge",
        "LinearEdge",
        "LinearEdge_bk",
        "ParabolicEdge",
        "ThresholdLinearEdge",
        "FlatLinearEdge",
        "DoubleLinearEdge",
    ],
    intervals=[10],
    interval_limits=[
        "percentile(0.01)",
        "percentile(99.99)",
    ],  # Interval filtering
    percentiles=[2, 5],
    percentile_limits=[None, 100],
)

In [ ]:
run_with_variable_percentiles(
    "albedo",
    albedo,
    lst,
    position="both",
    models=[
        "FlatRegressionEdge",
        "LinearEdge",
        "LinearEdge_bk",
        "ParabolicEdge",
        "ThresholdLinearEdge",
        "FlatLinearEdge",
        "DoubleLinearEdge",
    ],
    interval_nb=10,
    interval_limits=[
        "percentile(0.01)",
        "percentile(99.99)",
    ],  # Interval filtering
    percentile_bounds=[10, 0.01],
    percentile_intervals=[100, 1000000],
)